In [ ]:
from pathlib import Path
import pandas as pd

# Data Ingestion

Mounting to your google drive. When I share the folder with you it will be in your "Shared with me" folder. So, it will be helpful to create a shortcut to this folder in your "My Drive" folder. This way the below paths will still be accurate.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
data = '/content/drive/My Drive/Capstone/Data'
flowData = '/content/drive/My Drive/Capstone/Data/Flow'
packetData = '/content/drive/My Drive/Capstone/Data/Packet'

In [ ]:
for file in Path(flowData).iterdir():
  print(file)

/content/drive/My Drive/Capstone/Data/Flow/DDoS-HTTP_Flood-.pcap_Flow.csv
/content/drive/My Drive/Capstone/Data/Flow/DoS-HTTP_Flood.pcap_Flow.csv
/content/drive/My Drive/Capstone/Data/Flow/DoS-HTTP_Flood1.pcap_Flow.csv
/content/drive/My Drive/Capstone/Data/Flow/DNS_Spoofing.pcap_Flow.csv
/content/drive/My Drive/Capstone/Data/Flow/XSS.pcap_Flow.csv
/content/drive/My Drive/Capstone/Data/Flow/DictionaryBruteForce.pcap_Flow.csv
/content/drive/My Drive/Capstone/Data/Flow/BenignTraffic.pcap_Flow.csv
/content/drive/My Drive/Capstone/Data/Flow/BenignTraffic1.pcap_Flow.csv
/content/drive/My Drive/Capstone/Data/Flow/BenignTraffic2.pcap_Flow.csv
/content/drive/My Drive/Capstone/Data/Flow/BenignTraffic3.pcap_Flow.csv


In [ ]:
def fileCombiner(filesToCombine:list):
  dfToCombine = []
  for file in filesToCombine:
    csv = pd.read_csv(file)
    dfToCombine.append(csv)
  return pd.concat(dfToCombine)

In [ ]:
benignFlow = fileCombiner(['/content/drive/My Drive/Capstone/Data/Flow/BenignTraffic.pcap_Flow.csv','/content/drive/My Drive/Capstone/Data/Flow/BenignTraffic1.pcap_Flow.csv','/content/drive/My Drive/Capstone/Data/Flow/BenignTraffic2.pcap_Flow.csv','/content/drive/My Drive/Capstone/Data/Flow/BenignTraffic3.pcap_Flow.csv'])
ddosFlow = fileCombiner(['/content/drive/My Drive/Capstone/Data/Flow/DDoS-HTTP_Flood-.pcap_Flow.csv'])
dosFlow = fileCombiner(['/content/drive/My Drive/Capstone/Data/Flow/DoS-HTTP_Flood.pcap_Flow.csv', '/content/drive/My Drive/Capstone/Data/Flow/DoS-HTTP_Flood1.pcap_Flow.csv'])
dnsFlow = fileCombiner(['/content/drive/My Drive/Capstone/Data/Flow/DNS_Spoofing.pcap_Flow.csv'])
xssFlow = fileCombiner(['/content/drive/My Drive/Capstone/Data/Flow/XSS.pcap_Flow.csv'])
bruteFORCEFlow = fileCombiner(['/content/drive/My Drive/Capstone/Data/Flow/DictionaryBruteForce.pcap_Flow.csv'])
allFlowData = {'benignFlow':benignFlow, 'ddosFlow':ddosFlow, 'dosFlow':dosFlow, 'dnsFlow':dnsFlow, 'xssFlow':xssFlow, 'bruteFORCEFlow':bruteFORCEFlow}


# Data Pre Processing

## Phase 1: Data Preparation and Sampling
###Task 1.1: Random Dataset Generation Function
Write a Python function that samples the packet-level dataset randomly according to these speci-fications:
 - Benign traffic: 200,000 rows (roughly 97–98% of the sample).
 - Attack traffic: 4,000–6,200 rows (roughly 2–3% of the sample).
 - Attack traffic should be randomly distributed across the five attack types listed above.
 - Randomization must produce a different composition on each run. A configurable random
seed is a good idea.
 - Include basic validation and integrity checks.
###4.1.2 Task 1.2: Data Preprocessing
Prepare the data so the downstream models have a fair chance:
 - Handle missing values and outliers.
 - Apply scaling or normalization where it helps.
 - Use feature selection or dimensionality reduction if it improves results.
 - Address any quality issues specific to network traffic data.
 - Document every preprocessing choice with a short justification.

#Phase 2: Unsupervised Learning for Initial Detection (Anomaly-Based IDS)

##Task 2.1: Packet-Level Anomaly Detection
Build an unsupervised model that separates benign from non-benign packets.
 - Any unsupervised method is fine. Autoencoders and k-means are reasonable starting points,
and you can combine them: for example, train an autoencoder, take its reconstruction loss as
a feature, then cluster the result with k-means so the clusters separate benign from non-benign
groups. Use whatever you can justify.
 - Do not use packet labels during training.
 - Tune hyperparameters systematically where applicable.
 - Generate initial alerts based on anomaly scores or cluster assignments.
###4.2.2 Task 2.2: Alert Generation and Analysis
 - Pick thresholds for the packet-level anomaly detector and justify them.
 - Analyze the balance between false positives and false negatives.
 - Report initial detection performance.

 ## Evaluation and Metrics
 - Precision, recall, and F1-score for overall detection.
 - Per-attack detection rate for each of the five attack types.
 - False Positive Rate (FPR) and False Negative Rate (FNR).
 - AUC-ROC.
 - Confusion matrix.
 - A short analysis of how you handled the imbalance in the data.

# Phase 3: Supervised Learning for False Positive Reduction (Signature-Based IDS)

Every packet in the packet-level dataset has corresponding flow data in the flow-level dataset, identified by a flow ID of the form
``` [bash]
[source IP]-[destination IP]-[sender port]-[receiver port]
```

The flow-level dataset may contain multiple rows for a single flow, because flows longer than 2 minutes are automatically split into successive 2-minute segments that share the same flow ID. When you process flow data, group by flow ID and use the right aggregation per feature (mean, sum, or max, depending on what the feature means) to produce one unified record per flow. You will need to identify the flow-level rows that correspond to the packet-level dataset generated in Phase 2. You should also generate a second random dataset directly from the flow-level data, using the same proportions (200,000 benign rows and 4,000–6,200 non-benign rows). This flow-level dataset is what you will use for the supervised stage. Figures 1 and 2 (in the original PDF from Dr. Ardeshir) show the correspondence between a packet-level row and the flow-level rows derived from it.

## Task 3.1: Flow-Level Anomaly Detection (Optional but Recommended)
Optional enhancement. Apply the same anomaly-based approach you used in Phase 2 to the
flow-level data.
 - Run your unsupervised method on the flow-level dataset, using the flows that correspond to the data points you selected at the packet level.
 - Use what you learn from this to inform the supervised stage that follows.
 - Compare packet-level and flow-level unsupervised performance. Test the hypothesis that flow-based features perform better in the unsupervised setting and report what you find.
 ## Task 3.2: Flow-Based Feature Engineering
 - For each packet flagged as anomalous in Phase 2, locate the corresponding flow-level data.
 - Generate a new random dataset over flow-based features using the same sampling function
from Task 1.1.
 - Remember the flow segmentation behaviour described above: group rows by flow ID and
apply per-feature aggregations to produce one record per flow.
 - If you completed Task 3.1, use its findings to guide feature selection and engineering.
## Task 3.3: Signature-Based IDS Implementation
Build a supervised model that re-evaluates the alerts from Phase 2.
 - Use any model you can defend: Graph Neural Networks, Random Forest, SVM, deep neural
networks, XGBoost, ensembles, or something else. Pick what fits your data.
 - If you completed Task 3.1, you can incorporate features derived from flow-level anomaly
detection. Run the supervised model with and without those features so the contribution is
clear.
 - Use proper train/validation/test splits.
 - Use the trained model to re-classify the packets initially flagged in Phase 2.
 - Aim to reduce false positives while preserving detection accuracy.

 ## Evaluation and Metrics
 - Percentage reduction in false positives compared to Phase 2.
 - Overall system accuracy after the two stages run together.
 - Class-level evaluation and performance report.
 - Precision and recall for the combined system.
 - Computational overhead.
 - Time-complexity comparison between the two phases.



# Comparative Analysis
 - How does your two-stage system compare to single-stage approaches?
 - How does performance vary across attack types?
 - Are your improvements statistically significant?
 - Which attack types can be detected well using packet-level detection alone, and why do you
think that is?
 - How do DDoS and DoS look in your unsupervised clusters at the packet level (and at the
flow level, if you implemented Task 3.1)?